# Stage 4 — Link recalls to devices

Parse the frozen recall records and expand their **structured `k_numbers` field** to individual
devices. This links recalls to devices by identity — never by fuzzy name matching.

Network: none — reads `snapshot/recall_raw.json.gz`.

In [ ]:
import gzip, json
import pandas as pd

with gzip.open("snapshot/recall_raw.json.gz", "rt") as f:
    recalls = json.load(f)
corpus = pd.read_csv("data/corpus.csv", dtype=str)
VALID = set(corpus["k_number"])
print(f"{len(recalls)} recall records loaded")

### Expand k_numbers → device-level recall links

In [ ]:
links = []
for rec in recalls:
    kns = rec.get("k_numbers") or []
    for k in kns:
        k = str(k).strip().upper()
        links.append({
            "k_number": k,
            "product_res_number": rec.get("product_res_number"),
            "reason_for_recall": rec.get("reason_for_recall"),
            "event_date_initiated": rec.get("event_date_initiated"),
            "recalling_firm": rec.get("recalling_firm"),
            "in_corpus": k in VALID,
        })
ldf = pd.DataFrame(links)
ldf.to_csv("data/recall_links.csv", index=False)

n_links = len(ldf)
n_with_kn = ldf["k_number"].notna().sum()
n_corpus_recalled = ldf[ldf["in_corpus"]]["k_number"].nunique()
print(f"CHECKPOINT  recall records {len(recalls)}")
print(f"CHECKPOINT  recall->device links {n_links} | linking to corpus devices {n_corpus_recalled}")